# Baseline: hierarkisk multi-head CNN (nivå 1 + nivå 2)

Denne notebooken bygger videre på de ferdige `train/val/test`-splittene og trener en enkel baseline-modell:

- **Nivå 1**: `Tesla` vs `Other` (binary)
- **Nivå 2**: Tesla-underklasser (sparse categorical), **maskert for `Other`** med `sample_weight`.

## Forutsetninger
- Du har `datasplitt/train.csv`, `val.csv`, `test.csv`.
- CSV-ene inneholder minst: `image`, `model`, `lighting`.
- `image` er relativ sti under `IMG_ROOT` (samme som i `prepare_dataset.py`).

## Steg i denne notebooken
- Leser datasplitt (train/val/test) fra `datasplitt/`.
- Forbereder labels og sample weights for hierarkisk trening.
- Bygger `tf.data` pipeline med robust bildekoding.
- Trener en enkel baseline-modell med to output-hoder (nivå 1 og nivå 2).
- Evaluerer totalt og per lyskategori.

### Imports og grunnkonfigurasjon

Importerer nødvendige biblioteker og setter grunnparametre (seed, bildeformat, batch-størrelse).

In [1]:
# 0) Imports
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

print("TensorFlow:", tf.__version__)


TensorFlow: 2.15.1


### Les datasplitt (train/val/test)

Finner prosjektroten og leser `train.csv`, `val.csv` og `test.csv` fra `datasplitt/`. Viser en rask oversikt over datasettet.

In [2]:
# 1) Konfig

# Vi bruker `annotations/combined_clean_onedrive.csv` som anker for å finne riktig prosjektrot.
# Splittene forventes å være lagret av test_split.ipynb i mappen `datasplitt/`.

SEED = 42
IMG_SIZE = (300, 300)   # Juster om dereen vil bruke annet format  (224, 224 er vanlig for mange pre-trente modeller)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

tf.random.set_seed(SEED)
np.random.seed(SEED)

# Finn prosjektrot robust (forutsetter at mappen 'annotations' finnes)
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in [cwd, *cwd.parents]
    if (p / "annotations" / "combined_clean_onedrive.csv").exists()
)

SPLIT_DIR = PROJECT_ROOT / "datasplitt"
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV   = SPLIT_DIR / "val.csv"
TEST_CSV  = SPLIT_DIR / "test.csv"

if not (TRAIN_CSV.exists() and VAL_CSV.exists() and TEST_CSV.exists()):
    raise FileNotFoundError("Finner ikke train/val/test i datasplitt/.")

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLIT_DIR:", SPLIT_DIR)
print("rows train/val/test:", len(train_df), len(val_df), len(test_df))
print("columns:", train_df.columns.tolist())
display(train_df.head())

PROJECT_ROOT: C:\Users\andre\DAT191\visual-vehicle-recognition-varying-lighting-conditions
SPLIT_DIR: C:\Users\andre\DAT191\visual-vehicle-recognition-varying-lighting-conditions\datasplitt
rows train/val/test: 3033 651 651
columns: ['color', 'image', 'lighting', 'model', 'source', 'year', 'lvl1', 'lvl2', 'strat_key']


,color,image,lighting,model,source,year,lvl1,lvl2,strat_key
0,Light gray/Silver,Eksternt/Tesla/daylight/Kaggle/S/images (53).jpg,Light,S 2012–2015,external,2012–2015,Tesla,S 2012–2015,Tesla|S 2012–2015|Light
1,Light gray/Silver,Eksternt/non-tesla/daylight/0814_01435.jpg,Light,Other car,external,NaN,Other,NaN,Other|NA|Light
2,Light gray/Silver,Eksternt/non-tesla/daylight/1804_00995.jpg,Light,Other car,external,NaN,Other,NaN,Other|NA|Light
3,Blue,Egenprodusert/Tesla/low-light/%~n1-119.jpg,Medium,3 2017–2023,internal,2017–2023,Tesla,3 2017–2023,Tesla|3 2017–2023|Medium
4,Blue,Egenprodusert/Tesla/daylight/20260129_12224045...,Light,Y 2025-nå,internal,2025-nå,Tesla,Y 2025-nå,Tesla|Y 2025-nå|Light


### Angi bildemappe (IMG_ROOT)

Setter rotmappen der bildefilene ligger. Stiene i `image`-kolonnen i CSV er **relative** til denne mappen.

In [7]:
# 2) Last split CSV-er

# Viktig: `IMG_ROOT / <image>` må peke til en faktisk fil på disk.
# Eksempelstier i CSV kan være som: Eksternt/Tesla/... eller Egenprodusert/...
# 2) Sett IMG_ROOT (datasett-roten som 'image' er relativ til)
IMG_ROOT = Path(r"C:\Users\andre\OneDrive - Høgskulen på VestlandEt\Johannes Kaste sine filer - Dat191-shared\Datasett\felles-sladdet")
print("IMG_ROOT exists:", IMG_ROOT.exists())

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

display(train_df.head())
print("train/val/test:", len(train_df), len(val_df), len(test_df))
print("columns:", train_df.columns.tolist())

IMG_ROOT exists: True


,color,image,lighting,model,source,year,lvl1,lvl2,strat_key
0,Light gray/Silver,Eksternt/Tesla/daylight/Kaggle/S/images (53).jpg,Light,S 2012–2015,external,2012–2015,Tesla,S 2012–2015,Tesla|S 2012–2015|Light
1,Light gray/Silver,Eksternt/non-tesla/daylight/0814_01435.jpg,Light,Other car,external,NaN,Other,NaN,Other|NA|Light
2,Light gray/Silver,Eksternt/non-tesla/daylight/1804_00995.jpg,Light,Other car,external,NaN,Other,NaN,Other|NA|Light
3,Blue,Egenprodusert/Tesla/low-light/%~n1-119.jpg,Medium,3 2017–2023,internal,2017–2023,Tesla,3 2017–2023,Tesla|3 2017–2023|Medium
4,Blue,Egenprodusert/Tesla/daylight/20260129_12224045...,Light,Y 2025-nå,internal,2025-nå,Tesla,Y 2025-nå,Tesla|Y 2025-nå|Light


train/val/test: 3033 651 651
columns: ['color', 'image', 'lighting', 'model', 'source', 'year', 'lvl1', 'lvl2', 'strat_key']


### Valider hierarkiske kolonner

Sjekker at splittene inneholder `lvl1` og `lvl2` (de skal være generert i `test_split.ipynb`).

In [8]:
# 3) Bygg/valider hierarki-kolonner (idempotent: ok om de allerede finnes)
def ensure_hierarchy(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for c in ["image", "model", "lighting"]:
        if c not in out.columns:
            raise KeyError(f"Mangler kolonne '{c}' i split-CSV. Fant: {out.columns.tolist()}")

    out["model"] = out["model"].astype("string")
    out["lighting"] = out["lighting"].astype("string")

    # lvl1: Tesla vs Other
    if "lvl1" not in out.columns:
        is_other = out["model"].str.strip().str.lower().eq("other")
        out["lvl1"] = np.where(is_other, "Other", "Tesla")

    # lvl2: Tesla-underklasse (bruk 'model' direkte, siden prepare_dataset.py periodiserer)
    if "lvl2" not in out.columns:
        out["lvl2"] = out["model"].where(out["lvl1"].eq("Tesla"), "NA")

    # sample weight for lvl2 (kun Tesla)
    if "w_lvl2" not in out.columns:
        out["w_lvl2"] = (out["lvl1"].eq("Tesla")).astype(np.float32)

    return out

train_df = ensure_hierarchy(train_df)
val_df   = ensure_hierarchy(val_df)
test_df  = ensure_hierarchy(test_df)

print("lvl1 counts (train):\n", train_df["lvl1"].value_counts())
print("\nlvl2 counts (train):\n", train_df["lvl2"].value_counts().head(20))
print("\nlighting counts (train):\n", train_df["lighting"].value_counts())

lvl1 counts (train):
 lvl1
Tesla    1597
Other    1436
Name: count, dtype: int64

lvl2 counts (train):
 lvl2
Y 2020–2024    600
3 2017–2023    241
Y 2025-nå      227
X              186
3 2024–nå      172
S 2016–nå       95
S 2012–2015     76
Name: count, dtype: int64

lighting counts (train):
 lighting
Light     2066
Dark       492
Medium     475
Name: count, dtype: Int64


### Sample weight for nivå 2 (w_lvl2)

Sikrer at `w_lvl2` finnes. Denne brukes til å maskere nivå‑2‑loss/metrics for `Other` (nivå 2 er ikke relevant når lvl1=Other).

In [9]:
# 4) Label-encoding

# `w_lvl2` brukes som sample_weight for nivå 2:
# - Tesla: w_lvl2=1.0 (nivå 2 teller i loss/metrics)
# - Other: w_lvl2=0.0 (nivå 2 ignoreres, siden hierarkiet stopper på nivå 1)

# lvl1: Other=0, Tesla=1 (fast)
lvl1_map = {"Other": 0, "Tesla": 1}

# lvl2: kun Tesla-klasser (fra train)
lvl2_classes = sorted(train_df.loc[train_df["lvl1"].eq("Tesla"), "lvl2"].astype("string").unique().tolist())
lvl2_to_id = {c: i for i, c in enumerate(lvl2_classes)}
num_lvl2 = len(lvl2_classes)

print("Antall lvl2-klasser:", num_lvl2)
print("Eksempel lvl2-klasser:", lvl2_classes[:10])

def add_encoded(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["y_lvl1"] = out["lvl1"].map(lvl1_map).astype(np.int32)

    # dummy 0 for Other (maskeres bort med w_lvl2=0)
    y2 = np.zeros(len(out), dtype=np.int32)
    tesla_mask = out["lvl1"].eq("Tesla").to_numpy()
    y2[tesla_mask] = out.loc[tesla_mask, "lvl2"].map(lvl2_to_id).astype(np.int32).to_numpy()
    out["y_lvl2"] = y2

    out["w_lvl1"] = 1.0
    out["w_lvl2"] = out["w_lvl2"].astype(np.float32)
    return out

train_df = add_encoded(train_df)
val_df   = add_encoded(val_df)
test_df  = add_encoded(test_df)

display(train_df[["image","lvl1","lvl2","lighting","y_lvl1","y_lvl2","w_lvl2"]].head())

Antall lvl2-klasser: 7
Eksempel lvl2-klasser: ['3 2017–2023', '3 2024–nå', 'S 2012–2015', 'S 2016–nå', 'X', 'Y 2020–2024', 'Y 2025-nå']


,image,lvl1,lvl2,lighting,y_lvl1,y_lvl2,w_lvl2
0,Eksternt/Tesla/daylight/Kaggle/S/images (53).jpg,Tesla,S 2012–2015,Light,1,2,1.0
1,Eksternt/non-tesla/daylight/0814_01435.jpg,Other,NaN,Light,0,0,0.0
2,Eksternt/non-tesla/daylight/1804_00995.jpg,Other,NaN,Light,0,0,0.0
3,Egenprodusert/Tesla/low-light/%~n1-119.jpg,Tesla,3 2017–2023,Medium,1,0,1.0
4,Egenprodusert/Tesla/daylight/20260129_12224045...,Tesla,Y 2025-nå,Light,1,6,1.0


### Label‑encoding for trening

Konverterer tekstlabelene (`lvl1`, `lvl2`) til heltalls‑targets (`y_lvl1`, `y_lvl2`) som kreves av Keras, og beholder `w_lvl2` for maskering.

In [10]:
# 5) Sjekk at noen bildefiler faktisk finnes (sanity)

# Keras forventer heltalls-labels for sparse_categorical_crossentropy.
# Vi lager derfor y_lvl1 (0/1) og y_lvl2 (0..K-1).
# For Other-rader settes y_lvl2 til en dummyverdi (0), men disse maskeres bort via w_lvl2=0.

sample_paths = [str(IMG_ROOT / p) for p in train_df["image"].astype(str).head(10)]
exists = [Path(p).exists() for p in sample_paths]
for p, e in zip(sample_paths, exists):
    print(e, p)

True C:\Users\andre\OneDrive - Høgskulen på VestlandEt\Johannes Kaste sine filer - Dat191-shared\Datasett\felles-sladdet\Eksternt\Tesla\daylight\Kaggle\S\images (53).jpg
True C:\Users\andre\OneDrive - Høgskulen på VestlandEt\Johannes Kaste sine filer - Dat191-shared\Datasett\felles-sladdet\Eksternt\non-tesla\daylight\0814_01435.jpg
True C:\Users\andre\OneDrive - Høgskulen på VestlandEt\Johannes Kaste sine filer - Dat191-shared\Datasett\felles-sladdet\Eksternt\non-tesla\daylight\1804_00995.jpg
True C:\Users\andre\OneDrive - Høgskulen på VestlandEt\Johannes Kaste sine filer - Dat191-shared\Datasett\felles-sladdet\Egenprodusert\Tesla\low-light\%~n1-119.jpg
True C:\Users\andre\OneDrive - Høgskulen på VestlandEt\Johannes Kaste sine filer - Dat191-shared\Datasett\felles-sladdet\Egenprodusert\Tesla\daylight\20260129_122240454_iOS.jpg
True C:\Users\andre\OneDrive - Høgskulen på VestlandEt\Johannes Kaste sine filer - Dat191-shared\Datasett\felles-sladdet\Egenprodusert\Tesla\daylight\IMG_1367.jp

### Bygg tf.data pipeline (robust dekoding)

Lager `tf.data.Dataset` for train/val/test. Bilder lastes med Pillow via `tf.py_function` for å støtte bl.a. MPO/WEBP. MPO leses som første frame.

In [11]:
# 6) tf.data pipeline

# NB: `tf.py_function` kjører Python-kode (Pillow) og er typisk CPU-bundet.
# Det er robust for ulike bildeformater (MPO/WEBP), men kan være tregere enn ren TF-dekoding.
# `set_shape` er viktig slik at Keras kjenner statiske dimensjoner videre i grafen.

def decode_and_resize(path: tf.Tensor) -> tf.Tensor:
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)  # jpg/png/webp OK
    img = tf.image.resize(img, IMG_SIZE, method="bilinear")
    img = tf.cast(img, tf.float32) / 255.0
    return img

def make_dataset(df: pd.DataFrame, shuffle: bool) -> tf.data.Dataset:
    paths = np.array([str(IMG_ROOT / p) for p in df["image"].astype(str).to_list()], dtype=np.str_)
    y1 = df["y_lvl1"].to_numpy(np.int32)
    y2 = df["y_lvl2"].to_numpy(np.int32)
    w1 = df["w_lvl1"].to_numpy(np.float32)
    w2 = df["w_lvl2"].to_numpy(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, y1, y2, w1, w2))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df), 5000), seed=SEED, reshuffle_each_iteration=True)

    def _map(path, y1, y2, w1, w2):
        img = decode_and_resize(path)
        y = {"lvl1": y1, "lvl2": y2}
        sw = {"lvl1": w1, "lvl2": w2}   # lvl2 maskeres for Other via w2=0
        return img, y, sw

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df, shuffle=False)
test_ds  = make_dataset(test_df, shuffle=False)

for x, y, sw in train_ds.take(1):
    print("x:", x.shape, x.dtype)
    print("y lvl1:", y["lvl1"].shape, "y lvl2:", y["lvl2"].shape)
    print("sw lvl1:", sw["lvl1"].shape, "sw lvl2:", sw["lvl2"].shape)

x: (32, 300, 300, 3) <dtype: 'float32'>
y lvl1: (32,) y lvl2: (32,)
sw lvl1: (32,) sw lvl2: (32,)


## Baseline-modell

Backbone er lik den spesifiserte (Conv2D/MaxPool ×4 + Flatten + Dropout). Deretter legges to Dense-hoder på toppen: `lvl1` (sigmoid) og `lvl2` (softmax).

### TensorBoard logging

Setter opp en unik loggmappe og en `TensorBoard`‑callback. (Hvis du vil logge under trening: legg callbacken inn i `callbacks` i treningscellen.)

In [13]:
# TensorBoard-oppsett: hver kjøring får en egen undermappe med timestamp.
# Tips: Start TensorBoard i terminal med:
# tensorboard --logdir <PROJECT_ROOT>/logs/fit --port 6006

# 7) Modell: backbone + 2 hoder
inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")

backbone = keras.Sequential(
    [
        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dropout(0.4),
    ],
    name="backbone",
)

x = backbone(inputs)

# Nivå 1 head
out_lvl1 = keras.layers.Dense(1, activation="sigmoid", name="lvl1")(x)

# Nivå 2 head
out_lvl2 = keras.layers.Dense(num_lvl2, activation="softmax", name="lvl2")(x)

model = keras.Model(inputs=inputs, outputs={"lvl1": out_lvl1, "lvl2": out_lvl2}, name="baseline_hierarchical_cnn")
model.summary()

Model: "baseline_hierarchical_cnn"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image (InputLayer)          [(None, 300, 300, 3)]        0         []                            
                                                                                                  
 backbone (Sequential)       (None, 16384)                112576    ['image[0][0]']               
                                                                                                  
 lvl1 (Dense)                (None, 1)                    16385     ['backbone[0][0]']            
                                                                                                  
 lvl2 (Dense)                (None, 7)                    114695    ['backbone[0][0]']            
                                                                          

## Compile

Begge hoder bruker Adam(6e-4). Loss/metrics settes per output:
- Nivå 1: `binary_crossentropy`, `binary_accuracy`
- Nivå 2: `sparse_categorical_crossentropy`, `accuracy`

Nivå 2 maskeres for `Other` via `sample_weight` (`w_lvl2=0`).

### Compile modell

Kompilerer modellen med Adam(6e‑4). Nivå 1 bruker `binary_crossentropy`, nivå 2 bruker `sparse_categorical_crossentropy`.

In [14]:
# 8) Compile (per-output)

# Merk: `lvl2`-accuracy her er standard (u-vektet) metric.
# Dersom du vil at lvl2-metrics skal maskeres av w_lvl2 i rapportering, bruk weighted_metrics i compile().

opt = keras.optimizers.Adam(learning_rate=6e-4)

model.compile(
    optimizer=opt,
    loss={
        "lvl1": "binary_crossentropy",
        "lvl2": "sparse_categorical_crossentropy",
    },
    metrics={
        "lvl1": ["binary_accuracy"],
        "lvl2": ["accuracy"],
    },
)

### Trening

Trener modellen på treningssettet med validering, EarlyStopping og (valgfritt) TensorBoard‑logging.

In [ ]:
# 9) Trening (baseline)
EPOCHS = 5

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## Evaluering

1) `model.evaluate` på hele testsettet (lvl2 vektet bort for Other).
2) Tesla-only test (mer intuitive nivå-2-tall).

### Kjør evaluering per lyskategori

Kjører prediksjon på **testsettet** og bygger en tabell med metrikker for hver lyskategori (3 rader).

In [ ]:
# Tabellen under beregnes på testsettet (test_df).
# Hvis du ønsker samme analyse på valideringssettet, bytt test_df -> val_df.

# 10) Eval på hele test (lvl2 masked via sample weights)
test_metrics = model.evaluate(test_ds, return_dict=True)
print(test_metrics)

### Kodecelle

Kjører et delsteg i pipeline.

In [ ]:
# 11) Tesla-only eval
tesla_test_df = test_df[test_df["lvl1"].eq("Tesla")].copy()
tesla_test_df["w_lvl2"] = 1.0  # alle teller på lvl2 her

tesla_test_ds = make_dataset(tesla_test_df, shuffle=False)
tesla_metrics = model.evaluate(tesla_test_ds, return_dict=True)
print(tesla_metrics)

### Lagre Keras modell



In [ ]:
# 12) Lagre modell (valgfritt)
OUT_DIR = PROJECT_ROOT / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "baseline_hierarchical_cnn.keras"

model.save(OUT_PATH)
print("Saved:", OUT_PATH)